# FIFA Standings (Single Season)

# This is for testing purposes

In [669]:
import os # Load environment variables from a .env file
import json # JSON parsing
import requests # Send HTTP requests
import pandas as pd # Lets us work in pandas dataframes
from mysql import connector # Allows to connect to MySQL databases
from dotenv import load_dotenv # Load environment variables from a .env file

In [670]:
load_dotenv() # Load environment variables from a .env file

True

In [671]:
API_KEY = os.getenv("API_KEY") # Get the API
API_HOST = os.getenv("API_HOST") # Get the API host
SEASON = 2022 # Set the season to 2022
LEAGUE_ID = 1 # Set the league ID to 1 (FIFA World Cup)


In [672]:
url = f"https://{API_HOST}/standings"

headers = {
    "x-apisports-key": API_KEY
}

querystring = {"league": LEAGUE_ID, "season": SEASON}

In [673]:
print(API_HOST)
print(SEASON)
print(LEAGUE_ID)
print(querystring)

v3.football.api-sports.io
2022
1
{'league': 1, 'season': 2022}


# EXTRACT

In [674]:
response = requests.get(url=url, headers=headers, params=querystring)

In [675]:
payload = response.json() # Parse the JSON response

In [676]:
payload

{'get': 'standings',
 'parameters': {'league': '1', 'season': '2022'},
 'errors': [],
 'results': 1,
 'paging': {'current': 1, 'total': 1},
 'response': [{'league': {'id': 1,
    'name': 'World Cup',
    'country': 'World',
    'logo': 'https://media.api-sports.io/football/leagues/1.png',
    'flag': None,
    'season': 2022,
    'standings': [[{'rank': 1,
       'team': {'id': 1118,
        'name': 'Netherlands',
        'logo': 'https://media.api-sports.io/football/teams/1118.png'},
       'points': 7,
       'goalsDiff': 4,
       'group': 'Group A',
       'form': 'WDW',
       'status': 'same',
       'description': 'Promotion - World Cup (Play Offs)',
       'all': {'played': 3,
        'win': 2,
        'draw': 1,
        'lose': 0,
        'goals': {'for': 5, 'against': 1}},
       'home': {'played': None,
        'win': None,
        'draw': None,
        'lose': None,
        'goals': {'for': None, 'against': None}},
       'away': {'played': None,
        'win': None,
      

In [677]:
formatted_response = json.dumps(payload, indent=4) # Format the JSON response for readability
print(formatted_response)

{
    "get": "standings",
    "parameters": {
        "league": "1",
        "season": "2022"
    },
    "errors": [],
    "results": 1,
    "paging": {
        "current": 1,
        "total": 1
    },
    "response": [
        {
            "league": {
                "id": 1,
                "name": "World Cup",
                "country": "World",
                "logo": "https://media.api-sports.io/football/leagues/1.png",
                "flag": null,
                "season": 2022,
                "standings": [
                    [
                        {
                            "rank": 1,
                            "team": {
                                "id": 1118,
                                "name": "Netherlands",
                                "logo": "https://media.api-sports.io/football/teams/1118.png"
                            },
                            "points": 7,
                            "goalsDiff": 4,
                            "group": "Group 

# TRANSFORM

In [678]:
standings_list = payload['response'][0]['league']['standings']
formatted_standings_list = json.dumps(standings_list, indent=4) # Format the standings list for readability
print(formatted_standings_list)

[
    [
        {
            "rank": 1,
            "team": {
                "id": 1118,
                "name": "Netherlands",
                "logo": "https://media.api-sports.io/football/teams/1118.png"
            },
            "points": 7,
            "goalsDiff": 4,
            "group": "Group A",
            "form": "WDW",
            "status": "same",
            "description": "Promotion - World Cup (Play Offs)",
            "all": {
                "played": 3,
                "win": 2,
                "draw": 1,
                "lose": 0,
                "goals": {
                    "for": 5,
                    "against": 1
                }
            },
            "home": {
                "played": null,
                "win": null,
                "draw": null,
                "lose": null,
                "goals": {
                    "for": null,
                    "against": null
                }
            },
            "away": {
                "played"

In [679]:
rows = []
column_names = ['season', 'position', 'team_id', 'team', 'played', 'won', 'draw', 'lost', 'goals_for', 'goals_against', 'goal_diff', 'points', 'form']

for group in standings_list:
    for team in group:
        row = {
            'season':           SEASON,
            'position':         team['rank'],
            'team_id':          team['team']['id'],
            'team':             team['team']['name'],
            'played':           team['all']['played'],
            'won':              team['all']['win'],
            'draw':             team['all']['draw'],
            'lost':             team['all']['lose'],
            'goals_for':        team['all']['goals']['for'],
            'goals_against':    team['all']['goals']['against'],
            'goal_diff':        team['goalsDiff'],
            'points':           team['points'],
            'form':             team['form']
        }
        rows.append(row)

In [680]:
print(rows)

[{'season': 2022, 'position': 1, 'team_id': 1118, 'team': 'Netherlands', 'played': 3, 'won': 2, 'draw': 1, 'lost': 0, 'goals_for': 5, 'goals_against': 1, 'goal_diff': 4, 'points': 7, 'form': 'WDW'}, {'season': 2022, 'position': 2, 'team_id': 13, 'team': 'Senegal', 'played': 3, 'won': 2, 'draw': 0, 'lost': 1, 'goals_for': 5, 'goals_against': 4, 'goal_diff': 1, 'points': 6, 'form': 'WWL'}, {'season': 2022, 'position': 3, 'team_id': 2382, 'team': 'Ecuador', 'played': 3, 'won': 1, 'draw': 1, 'lost': 1, 'goals_for': 4, 'goals_against': 3, 'goal_diff': 1, 'points': 4, 'form': 'LDW'}, {'season': 2022, 'position': 4, 'team_id': 1569, 'team': 'Qatar', 'played': 3, 'won': 0, 'draw': 0, 'lost': 3, 'goals_for': 1, 'goals_against': 7, 'goal_diff': -6, 'points': 0, 'form': 'LLL'}, {'season': 2022, 'position': 1, 'team_id': 10, 'team': 'England', 'played': 3, 'won': 2, 'draw': 1, 'lost': 0, 'goals_for': 9, 'goals_against': 2, 'goal_diff': 7, 'points': 7, 'form': 'WDW'}, {'season': 2022, 'position': 2

In [681]:
df = pd.DataFrame(rows, columns=column_names)

In [682]:
df.head(20)

,season,position,team_id,team,played,won,draw,lost,goals_for,goals_against,goal_diff,points,form
0,2022,1,1118,Netherlands,3,2,1,0,5,1,4,7,WDW
1,2022,2,13,Senegal,3,2,0,1,5,4,1,6,WWL
2,2022,3,2382,Ecuador,3,1,1,1,4,3,1,4,LDW
3,2022,4,1569,Qatar,3,0,0,3,1,7,-6,0,LLL
4,2022,1,10,England,3,2,1,0,9,2,7,7,WDW
5,2022,2,2384,USA,3,1,2,0,2,1,1,5,WDD
6,2022,3,22,Iran,3,1,0,2,4,7,-3,3,LWL
7,2022,4,767,Wales,3,0,1,2,1,6,-5,1,LLD
8,2022,1,26,Argentina,3,2,0,1,5,2,3,6,WWL
9,2022,2,24,Poland,3,1,1,1,2,2,0,4,LWD


In [683]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   season         32 non-null     int64
 1   position       32 non-null     int64
 2   team_id        32 non-null     int64
 3   team           32 non-null     str  
 4   played         32 non-null     int64
 5   won            32 non-null     int64
 6   draw           32 non-null     int64
 7   lost           32 non-null     int64
 8   goals_for      32 non-null     int64
 9   goals_against  32 non-null     int64
 10  goal_diff      32 non-null     int64
 11  points         32 non-null     int64
 12  form           32 non-null     str  
dtypes: int64(11), str(2)
memory usage: 3.4 KB


# LOAD

In [684]:
MYSQL_HOST = os.getenv("MYSQL_HOST") # Get the MySQL host
MYSQL_PORT = os.getenv("MYSQL_PORT") # Get the MySQL port
MYSQL_USER = os.getenv("MYSQL_USER") # Get the MySQL user
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD") # Get the MySQL password   
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE") # Get the MySQL database name



In [685]:
server_connection = connector.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    connection_timeout=10,
    autocommit=False,
    raise_on_warnings=True
)

server_cursor = server_connection.cursor()
print("Connected successfully!") if server_connection.is_connected() else print("Connection failed")

Connected successfully!


In [686]:
server_cursor.close()
server_connection.close()

In [687]:
from dotenv import load_dotenv
import os
import mysql.connector as connector

load_dotenv(".env", override=True)

MYSQL_HOST = os.getenv("MYSQL_HOST")
MYSQL_PORT = os.getenv("MYSQL_PORT")
MYSQL_USER = os.getenv("MYSQL_USER")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD")
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE")

print("Using database:", MYSQL_DATABASE)  # sanity check

db_connection = connector.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    database=MYSQL_DATABASE,
)

cursor = db_connection.cursor()
print("Connected successfully!") if db_connection.is_connected() else print("Connection failed")

Using database: worldcup_league_db
Connected successfully!


In [688]:
sql_table = "standings"
cursor.execute("SHOW TABLES LIKE %s",(f"{sql_table}",))

if cursor.fetchone() is None:
    print(f"Table '{sql_table}' does not exist.")
else:
    print(f"Table '{sql_table}' exists.")

Table 'standings' exists.


In [689]:
df = df.rename(columns={'goal_diff': 'goals_diff'})

table_cols = ['season', 'position', 'team_id', 'team', 'played', 'won', 'draw', 'lost', 'goals_for', 'goals_against', 'goals_diff', 'points', 'form']

standings_df = df[table_cols]

In [690]:
standings_records_tuples = standings_df.itertuples(index =False, name=None)  # Convert the DataFrame to a list of tuples for insertion


list_of_standings_records = list(standings_records_tuples)  # Convert the iterator to a list

In [691]:
UPSERT_QUERY = f"""
INSERT INTO {sql_table} (season, position, team_id, team, played, won, draw, lost, goals_for, goals_against, goals_diff, points, form)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s) as src
ON DUPLICATE KEY UPDATE
    position                = src.position,
    team_id                 = src.team_id,
    team                    = src.team,
    played                  = src.played,
    won                     = src.won,
    draw                    = src.draw,
    lost                    = src.lost,
    goals_for               = src.goals_for,
    goals_against           = src.goals_against,
    goals_diff              = src.goals_diff,
    points                  = src.points,
    form                    = src.form
"""

In [692]:
cursor.execute("DESCRIBE standings")
for column in cursor.fetchall():
    print(column)

('season', 'int', 'NO', 'PRI', None, '')
('position', 'int', 'NO', '', None, '')
('team_id', 'int', 'NO', 'PRI', None, '')
('team', 'varchar(100)', 'NO', '', None, '')
('played', 'int', 'NO', '', None, '')
('won', 'int', 'NO', '', None, '')
('draw', 'int', 'NO', '', None, '')
('lost', 'int', 'NO', '', None, '')
('goals_for', 'int', 'NO', '', None, '')
('goals_against', 'int', 'NO', '', None, '')
('goals_diff', 'int', 'NO', '', None, '')
('points', 'int', 'NO', '', None, '')
('form', 'varchar(5)', 'NO', '', None, '')


In [693]:
print(df.columns.tolist())

['season', 'position', 'team_id', 'team', 'played', 'won', 'draw', 'lost', 'goals_for', 'goals_against', 'goals_diff', 'points', 'form']


In [694]:
no_of_rows_uploaded_to_mysql = len(list_of_standings_records)

In [695]:
try:
    cursor.executemany(UPSERT_QUERY, list_of_standings_records)
    db_connection.commit()  # Commit the transaction to save changes
    print(f"Successfully uploaded {no_of_rows_uploaded_to_mysql} rows to the '{sql_table}' table.")
except Exception as e:
    print(f"Error occurred while uploading data: {e}")

finally:
    cursor.close()
    db_connection.close()
    print("Database connection closed.")

Successfully uploaded 32 rows to the 'standings' table.
Database connection closed.


In [696]:
print("df columns:", df.columns.tolist())
print("table_cols:", table_cols)

df columns: ['season', 'position', 'team_id', 'team', 'played', 'won', 'draw', 'lost', 'goals_for', 'goals_against', 'goals_diff', 'points', 'form']
table_cols: ['season', 'position', 'team_id', 'team', 'played', 'won', 'draw', 'lost', 'goals_for', 'goals_against', 'goals_diff', 'points', 'form']


In [697]:
print(len(list_of_standings_records))
print(list_of_standings_records[:2])  # peek at first couple entries

32
[(2022, 1, 1118, 'Netherlands', 3, 2, 1, 0, 5, 1, 4, 7, 'WDW'), (2022, 2, 13, 'Senegal', 3, 2, 0, 1, 5, 4, 1, 6, 'WWL')]


# RESULTS